`(1) Env 환경변수`

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [3]:
import os
from pprint import pprint
from typing import List, Tuple

`(3) langfuse handler 설정`

In [4]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

https://huggingface.co/datasets/allganize/RAG-Evaluation-Dataset-KO

In [ ]:
import pandas as pd
from datasets import load_dataset

# 데이터셋 로드
ds = load_dataset("allganize/RAG-Evaluation-Dataset-KO")


# 여기서는 'test' 데이터만 추출해서 CSV로 저장해보겠습니다.
df = ds['test'].to_pandas()

# CSV 파일로 저장
df.to_csv("data/rag_test_data.csv", index=False, encoding='utf-8-sig')

print("데이터가 'rag_test_data.csv'로 저장되었습니다.")

데이터가 'rag_test_data.csv'로 저장되었습니다.


In [ ]:
# 1. 필수 라이브러리 임포트
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. 데이터 로드 (저장해두신 CSV 파일)
# 파일 경로가 맞는지 꼭 확인하세요!
df = pd.read_csv("data/rag_test_data.csv")

# 3. 청크 분할기 설정 (500자 단위, 50자 중첩)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50
)

# 4. 데이터셋을 청크로 변환 및 Document 객체 생성
documents_with_metadata = []

print("데이터 처리 시작...")

for idx, row in df.iterrows():

    content_with_qa = f"{row['target_answer']}"
    
    # 텍스트를 청크로 분할
    chunks = text_splitter.split_text(content_with_qa)
    
    for chunk in chunks:
        # Document 객체 생성 시 question과 domain을 모두 메타데이터로 관리
        doc = Document(
            page_content=chunk,
            metadata={
                "domain": row.get('domain', 'N/A'),
                "question": row.get('question', 'N/A'), # 질문도 메타데이터에 저장
                "file_name": row.get('target_file_name', 'N/A'),
                "page": row.get('target_page_no', 0)
            }
        )
        documents_with_metadata.append(doc)
# 5. 결과 확인
print(f"완료! 총 {len(documents_with_metadata)}개의 청크가 생성되었습니다.")

# 첫 번째 청크와 메타데이터 출력
print("\n[첫 번째 청크 확인]")
print(f"내용: {documents_with_metadata[0].page_content}")
print(f"메타데이터: {documents_with_metadata[0].metadata}")


데이터 처리 시작...
완료! 총 305개의 청크가 생성되었습니다.

[첫 번째 청크 확인]
내용: 시중은행, 지방은행, 인터넷은행 모두 은행업을 영위하기 위해서는 '은행법' 제8조에 근거해 금융위원회의 인가를 필요로 합니다. 그러나 시중은행, 지방은행, 인터넷은행의 인가 요건 및 절차에는 일부 차이가 존재합니다.

첫째, 최저자본금의 경우, 시중은행은 1,000억원이 필요하며, 지방은행과 인터넷은행은 250억원이 필요합니다. 

둘째, 비금융주력자 주식 보유한도의 차이가 있습니다. 시중은행의 경우 4%이며, 지방은행은 15%, 인터넷은행은 34%입니다.

셋째, 영업구역 및 영업방식도 차이가 존재합니다. 시중은행과 지방은행은 온라인과 오프라인을 모두 활용하며 전국 또는 일부 지역에서 영업할 수 있지만, 인터넷전문은행은 전국에서 영업하지만 오직 온라인으로만 이루어집니다.
메타데이터: {'domain': 'finance', 'question': '시중은행, 지방은행, 인터넷은행의 인가 요건 및 절차에 차이가 있는데 그 차이점은 무엇인가요?', 'file_name': '[별첨] 지방은행의 시중은행 전환시 인가방식 및 절차.pdf', 'page': '4'}


In [18]:
#각 청크에 대한
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. LLM 설정
llm = ChatOpenAI(model="gpt-4o-mini")

# 2. 맥락 요약 프롬프트 정의
prompt = ChatPromptTemplate.from_template("""
이 텍스트는 전체 문서의 일부입니다. 
이 조각이 담고 있는 핵심 정보(예: 시중/지방/인터넷은행의 자본금 비교, 주식 보유 한도, 인가 근거법 등)를 구체적으로 요약해주세요.
이 요약문은 검색 시스템이 정확한 정보를 찾기 위한 '검색 인덱스' 역할을 합니다.

텍스트 조각: {chunk_content}
요약문:
""")

chain = prompt | llm | StrOutputParser()

# 3. 각 청크에 맥락 보강
enriched_documents = []

for doc in documents_with_metadata:
    # LLM에게 요약 요청
    context_summary = chain.invoke({"chunk_content": doc.page_content})
    
    # [요약문] + [본문] 결합
    enriched_content = f"{context_summary}"
    
    # 새로운 Document 객체 생성
    enriched_doc = Document(
        page_content=enriched_content,
        metadata=doc.metadata
    )
    enriched_documents.append(enriched_doc)

print(f"총 {len(enriched_documents)}개의 청크에 맥락이 보강되었습니다.")
print("\n[보강된 청크 예시]:")
print(enriched_documents[0].page_content)

총 305개의 청크에 맥락이 보강되었습니다.

[보강된 청크 예시]:
- **인가 근거법**: 은행법 제8조에 근거하여 금융위원회의 인가 필요.
- **최저자본금**: 
  - 시중은행: 1,000억원
  - 지방은행: 250억원
  - 인터넷은행: 250억원
- **비금융주력자 주식 보유 한도**:
  - 시중은행: 4%
  - 지방은행: 15%
  - 인터넷은행: 34%
- **영업구역 및 방식**:
  - 시중은행 및 지방은행: 온라인과 오프라인, 전국 또는 일부 지역 영업
  - 인터넷은행: 전국 영업, 오직 온라인으로만 운영


In [ ]:
normal_db.delete_collection()
contextual_db.delete_collection()

In [30]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# 2. Chroma 벡터스토어 빌드 및 저장 
# 일반 문서용 벡터 DB (비교군)
normal_db = Chroma.from_documents(
    documents=documents_with_metadata,  # Replace with your document list
    embedding=embeddings,    
    collection_name="normal", 
    persist_directory="./chroma_db",
  
)
#보강된 문서용 벡터 DB (실험군)
contextual_db = Chroma.from_documents(
    documents=enriched_documents,  # Replace with your document list
    embedding=embeddings,    
    collection_name="contextural", 
    persist_directory="./chroma_db",
  
)


In [31]:

# 3. 테스트할 질문 3개
test_queries = [
    "시중은행의 최저자본금은 얼마야?",
    "대주주가 충족해야 하는 기타 요건",
    "은행업 인가를 받으려면 어떤 법을 근거로 해야 해?"
]

# 4. 성능 비교 실행
for query in test_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    normal_retriever = normal_db.as_retriever(search_kwargs={"k": 3})
    contextual_retriever = contextual_db.as_retriever(search_kwargs={"k": 3})
    # 일반 검색

    normal_results = normal_retriever.invoke(query)
    print(f"\n[일반 검색 결과]")
    for i, doc in enumerate(normal_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")


    
    # Contextual 검색
    contextual_results = contextual_retriever.invoke(query)
    print(f"\n[Contextual 검색 결과]")
    for i, doc in enumerate(contextual_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")


쿼리: '시중은행의 최저자본금은 얼마야?'

[일반 검색 결과]
  1. 시중은행, 지방은행, 인터넷은행 모두 은행업을 영위하기 위해서는 '은행법' 제8조에 근거해 금융위원회의 인가를 필요로 합니다. 그러나 시중은행, 지방은행, 인터넷은행의 인가 요건 ...
  2. 연체이자는 연체이자의 계산기간 동안 해당 연도마다 1월 1일 현재 전국은행이 적용하는 정기예금금리 중 가장 높은 금리의 2배에 해당하는 금리를 적용하여 산정한다. 이 경우 연체이자...
  3. 산업형기금이 3년 평균 9%로 가장 높은 수익률을 보이고 있습니다....

[Contextual 검색 결과]
  1. - **인가 근거법**: 은행법 제8조에 근거하여 금융위원회의 인가 필요.
- **최저자본금**: 
  - 시중은행: 1,000억원
  - 지방은행: 250억원
  - 인터넷은행:...
  2. 2023년 기준으로 시중은행, 지방은행, 인터넷은행의 자본금 비교 및 주식 보유 한도, 인가 근거법에 대한 정보가 포함되어 있습니다. 추가적인 자세한 내용은 문서 본문 참조 필요....
  3. 주요 정보: 
- 시중은행/지방은행/인터넷은행 자본금 비교: 시중은행 24.6%, 지방은행 4.8%
- 주식 보유 한도 및 인가 근거법: 구체적인 내용은 제공되지 않음.

이 정보...

쿼리: '대주주가 충족해야 하는 기타 요건'

[일반 검색 결과]
  1. 은행업을 신청하려면 대주주 요건을 충족해야 합니다. 대주주 요건으로는 부실금융기관 관련 책임이 없어야 하며, 주주구성계획이 은행법상 소유규제에 적합해야 합니다. 대주주 요건을 증명...
  2. 공통적으로 상정된 안건은 제무제표 승인, 정관 변경, 이사 선임, 이사 보수한도액 승인입니다. 2020년 정기 주주총회에서는 이익잉여금 처분계산서 승인, 감사 보수한도액 승인이 추...
  3. 2020년 3월 24일 주주총회에서 원고가 행사한 의결권은 제7기 제무제표, 이익잉여금 처분계산서 승인, 정관 변경, 이사 선임, 이사 보수한도액 승인, 감사 보수

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# 1. CrossEncoder 모델 객체 생성 (문자열 모델명을 여기에 전달)
encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

# 2. Reranker 객체 생성 (생성한 encoder 객체를 model 인자로 전달)
reranker = CrossEncoderReranker(model=encoder, top_n=3)

# 3. 압축 검색기 설정 (
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, 
    base_retriever=contextual_db.as_retriever(search_kwargs={"k": 5})  
)

# 4. 검색 실행
results = compression_retriever.invoke("시중은행의 최저자본금은 얼마야?")
for res in results:
    print(res.page_content)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

- **인가 근거법**: 은행법 제8조에 근거하여 금융위원회의 인가 필요.
- **최저자본금**: 
  - 시중은행: 1,000억원
  - 지방은행: 250억원
  - 인터넷은행: 250억원
- **비금융주력자 주식 보유 한도**:
  - 시중은행: 4%
  - 지방은행: 15%
  - 인터넷은행: 34%
- **영업구역 및 방식**:
  - 시중은행 및 지방은행: 온라인과 오프라인, 전국 또는 일부 지역 영업
  - 인터넷은행: 전국 영업, 오직 온라인으로만 운영
주요 정보: 
- 시중은행/지방은행/인터넷은행 자본금 비교: 시중은행 24.6%, 지방은행 4.8%
- 주식 보유 한도 및 인가 근거법: 구체적인 내용은 제공되지 않음.

이 정보는 검색 시스템에 있어 은행 자본금 비율 및 관련 법률 인지에 도움이 됩니다.
2023년 기준으로 시중은행, 지방은행, 인터넷은행의 자본금 비교 및 주식 보유 한도, 인가 근거법에 대한 정보가 포함되어 있습니다. 추가적인 자세한 내용은 문서 본문 참조 필요.
